In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import date

In [29]:
# Base URL for relative links
BASE_URL = "https://ncdc.gov.ng"

# Load saved HTML table
with open("/kaggle/input/ncdc-table/ncdc_table.html", "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

# Find all table rows 
rows = soup.find_all("tr")[1:]  # skip the first row (header)

# Store SN and PDF link
pdf_data = []

for row in rows:
    cols = row.find_all("td")
    if len(cols) >= 3:
        sn = cols[0].get_text(strip=True)
        title = cols[1].get_text(strip=True)
        link_tag = cols[2].find("a", href=True)
        if link_tag:
            pdf_url = urljoin(BASE_URL, link_tag['href'])
            pdf_data.append((sn, title, pdf_url))

# Print results
for sn, title, url in pdf_data[:9]:
    print(f"SN {sn} | Title: {title} | URL: {url}")

print(f"Total PDFs found: {len(pdf_data)}")

SN 1 | Title: An update of Lassa fever outbreak in Nigeria for Week 45 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/471dd6384cd4f288167c0881efee40ee.pdf
SN 2 | Title: An update of Lassa fever outbreak in Nigeria for Week 44 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/9910b5860f5b9c377cf14a8992d6905b.pdf
SN 3 | Title: An update of Lassa fever outbreak in Nigeria for Week 43 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/55b33161bbfe39fd72039808875e5c45.pdf
SN 4 | Title: An update of Lassa fever outbreak in Nigeria for Week 42 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/91f559d691c7cd411f7058f1496eef6f.pdf
SN 5 | Title: An update of Lassa fever outbreak in Nigeria for Week 41 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/68f0ee9000f3084b74ed892be0d3f79f.pdf
SN 6 | Title: An update of Lassa fever outbreak in Nigeria for Week 40 | URL: https://ncdc.gov.ng/themes/common/files/sitreps/ded728ef465f33739ad7928d327a2929.pdf
SN 7 | Title: An updat

In [31]:
# Helper functions 
def create_empty_pdf(filepath, text="Missing week"):
    c = canvas.Canvas(filepath)
    c.drawString(100, 750, text)
    c.save()

def weeks_in_year(year):
    """Return number of weeks in a given year (52 or 53)."""
    last_day = date(year, 12, 28)  # Dec 28 is always in the last ISO week of the year
    return last_day.isocalendar()[1]


In [ ]:
# Download or create PDFs

current_year = 2020
current_week = 1

for sn, title, pdf_url in pdf_data:
    # Extract week number from title
    week_match = re.search(r'Week (\d+)', title, re.IGNORECASE)
    if week_match:
        week_num = int(week_match.group(1))
    else:
        # fallback if week number not found in title
        week_num = current_week

    # Fill in missing weeks before this PDF
    while current_week < week_num:
        missing_filename = f"{current_year}-W{current_week:02d}_MISSING.pdf"
        missing_filepath = os.path.join(SAVE_DIR, missing_filename)
        create_empty_pdf(missing_filepath, f"Missing week {current_year}-W{current_week:02d}")
        print(f"Created placeholder: {missing_filename}")

        # Increment week
        current_week += 1
        max_weeks = weeks_in_year(current_year)
        if current_week > max_weeks:
            current_week = 1
            current_year += 1

    # Now handle current PDF
    filename = f"{current_year}-W{week_num:02d}.pdf"
    filepath = os.path.join(SAVE_DIR, filename)

    if pdf_url:
        try:
            r = requests.get(pdf_url)
            r.raise_for_status()
            with open(filepath, "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {filename}")
        except:
            # If download fails, create placeholder
            create_empty_pdf(filepath, f"Missing week {current_year}-W{week_num:02d}")
            print(f"Failed to download, created placeholder: {filename}")
    else:
        create_empty_pdf(filepath, f"Missing week {current_year}-W{week_num:02d}")
        print(f"No PDF link, created placeholder: {filename}")

    # Move to next week
    current_week = week_num + 1
    if current_week > 52:
        current_week = 1
        current_year += 1

print("All PDFs processed!")